## Imports


In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import random
import copy
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt
import optuna
from tqdm.auto import trange
from utils import Adam

from LINKS.Geometry import CurveEngine
from LINKS.Optimization import DifferentiableTools, Tools
from LINKS.CP import REFERENCE_POINTS, MAX_JOINTS, make_empty_submission, evaluate_submission
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PolynomialMutation
from pymoo.optimize import minimize
from pymoo.indicators.hv import HV

## Settings


In [ ]:
TARGET_PATH = Path('kangaroo_target_curves.npy')
STARTER_PATH = Path('starter_mechanism.npy')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = Path('BC_DS_JC_results') / RUN_ID

target_index = 1  # 0: round; 1: no ears/tail; 2: full kangaroo
TARGET_NAMES = ['Round kangaroo', 'Kangaroo without ears/tail', 'Full kangaroo']
SEED = 0
np.random.seed(SEED)
random.seed(SEED)

# GA minimizes mean point error + LAMBDA_MAX * maximum error, and material.
LAMBDA_MAX = 0.5
population_size = 100
num_generations = 1000
gd_ga_generations = 100

# GD minimizes one loss: w * distance / distance_limit + (1-w) * material / material_limit.
# 0.9 favors distance; 0.5 gives equal coefficients to the normalized objectives.
# The distance term uses Tools distance; the GA keeps its mean/max error objective.
GD_DISTANCE_WEIGHT = 0.9
gd_max_backtracks = 8
# Optuna tunes the learning rate with this weight held fixed.
tuning_steps = 1000
tuning_trials = 5
learning_rate_range = (1e-7, 1e-4)
SHOW_PLOTS = True  # Final target fit and GA/GD/GA objective comparison only.

## Load Data


In [ ]:
target_curves = np.load(TARGET_PATH)
if not 0 <= target_index < len(target_curves):
    raise ValueError('target_index must select one of the supplied target curves.')
target_curve = np.asarray(target_curves[target_index])
reference_point = np.asarray(REFERENCE_POINTS[target_index])

starter_mech = np.load(STARTER_PATH, allow_pickle=True).item()
x0 = np.asarray(starter_mech['x0'])
edges = np.asarray(starter_mech['edges'])
fixed_joints = np.asarray(starter_mech['fixed_joints'])
motor = np.asarray(starter_mech['motor'])
if len(x0) > MAX_JOINTS:
    raise ValueError(f'Mechanisms must have at most {MAX_JOINTS} joints.')

print(f'Problem {target_index + 1}: {TARGET_NAMES[target_index]}')
print(f'Scoring limits: distance < {reference_point[0]}, material < {reference_point[1]}')

## Compile Tools


In [ ]:
optimization_tools = Tools(device='cpu')
optimization_tools.compile()
PROBLEM_TOOLS = optimization_tools
PROBLEM_TOOLS.solver.compile()

COST_ENGINE = CurveEngine(resolution=PROBLEM_TOOLS.timesteps, normalize_scale=False)
COST_ENGINE.compile()

differentiable_optimization_tools = DifferentiableTools(device='cpu')
differentiable_optimization_tools.compile()

## GA Problem


In [ ]:
def mean_max_cost(point_errors, lambda_max):
    return point_errors.mean(axis=-1) + lambda_max * point_errors.max(axis=-1)


class mechanism_Synthesis_optimization(Problem):
    def __init__(self, edges, fixed_joints, motor, target_curve, reference_point,
                 lambda_max=LAMBDA_MAX):
        if not np.isfinite(lambda_max) or lambda_max < 0:
            raise ValueError("lambda_max must be finite and nonnegative")
        super().__init__(n_var=x0.shape[0] * 2, n_obj=2, n_ieq_constr=2, xl=0.0, xu=5.0)
        self.edges = edges
        self.fixed_joints = fixed_joints
        self.motor = motor
        self.N = x0.shape[0]
        self.target_curve = target_curve
        self.reference_point = np.asarray(reference_point)
        self.lambda_max = lambda_max
        # Match Tools' default output joint, even if sorting changes joint indices.
        batch = PROBLEM_TOOLS.get_preprocessed([x0], [edges], [fixed_joints], [motor])
        if not batch.valid[0]:
            raise ValueError("The fixed mechanism topology cannot be solved")
        self.target_idx = int(batch.orders[0][-1])

    def scoring_objectives(self, x):
        """Original Tools distance/material, for constraints and diagnostic plots."""
        x0s = list(np.asarray(x).reshape(-1, self.N, 2))
        distance, material = PROBLEM_TOOLS(
            x0s, [self.edges] * len(x0s), [self.fixed_joints] * len(x0s),
            [self.motor] * len(x0s), self.target_curve,
            target_idx=[self.target_idx] * len(x0s),
        )
        return np.column_stack((distance, material))

    def _evaluate(self, x, out, *args, **kwargs):
        original = self.scoring_objectives(x)
        x0s = list(x.reshape(-1, self.N, 2))
        # Use the original Tools scores for the scoring constraints.
        solutions = np.asarray(PROBLEM_TOOLS.solver(
            x0s, [self.edges] * len(x0s), [self.fixed_joints] * len(x0s),
            [self.motor] * len(x0s),
        ))
        valid = np.isfinite(solutions).all(axis=(1, 2, 3))
        traced_curves = solutions[:, self.target_idx].copy()
        # Avoid feeding failed simulations into alignment; reject them below.
        traced_curves[~valid] = self.target_curve
        aligned, target, _ = COST_ENGINE.optimal_alignment(traced_curves, self.target_curve)
        point_errors = np.linalg.norm(np.asarray(aligned) - np.asarray(target), axis=-1)
        cost = mean_max_cost(point_errors, self.lambda_max)
        valid &= np.isfinite(cost) & np.isfinite(original).all(axis=1)
        out["F"] = np.column_stack((np.where(valid, cost, np.inf), original[:, 1]))
        # Original scoring thresholds still apply, including the original 2*pi distance factor.
        out["G"] = np.where(valid[:, None], original - self.reference_point, np.inf)

## GD Function


In [ ]:
def combined_gd_loss(objectives, reference_point, distance_weight):
    """One weighted distance/material loss per mechanism."""
    scales = np.asarray(reference_point, dtype=float)
    weights = np.array([distance_weight, 1.0 - distance_weight])
    return (np.asarray(objectives, dtype=float) / scales) @ weights


def refine_gradient_population(
    x0, edges, fixed_joints, motor, target_curve, reference_point,
    *, distance_weight=0.9, optimizer='gradient_descent', learning_rate=1e-5,
    beta1=0.9, beta2=0.999, eps=1e-8, n_steps=1000, max_backtracks=8,
    bounds=(0.0, 5.0),
):
    """Refine both objectives with a fixed weighted loss and checked trial steps."""
    if optimizer not in {'gradient_descent', 'adam'}:
        raise ValueError("optimizer must be 'gradient_descent' or 'adam'")
    if not np.isfinite(distance_weight) or not 0 <= distance_weight <= 1:
        raise ValueError('distance_weight must be between 0 and 1.')
    if not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError('learning_rate must be finite and positive.')
    if n_steps < 0 or max_backtracks < 0:
        raise ValueError('Step and backtracking counts must be nonnegative.')
    scales = np.asarray(reference_point, dtype=float)
    if scales.shape != (2,) or not np.isfinite(scales).all() or (scales <= 0).any():
        raise ValueError('Both normalization scales must be finite and positive.')

    x = np.asarray(x0, dtype=float).copy()
    if len(x) == 0:
        return x
    if x.ndim != 3 or x.shape[-1] != 2:
        raise ValueError('x0 must have shape (population, joints, 2).')
    coefficients = np.array([distance_weight, 1.0 - distance_weight]) / scales

    def evaluate(population):
        count = len(population)
        distances, materials = optimization_tools(
            list(population), [edges] * count, [fixed_joints] * count,
            [motor] * count, target_curve, target_idx=None,
        )
        return np.column_stack((np.asarray(distances).reshape(-1),
                                np.asarray(materials).reshape(-1)))

    def valid(objectives):
        return np.isfinite(objectives).all(axis=1) & (objectives < scales).all(axis=1)

    objectives = evaluate(x)
    active = valid(objectives) & np.isfinite(x).all(axis=(1, 2))
    active &= ((x >= bounds[0]) & (x <= bounds[1])).all(axis=(1, 2))
    adams = ([Adam(learning_rate, beta1, beta2, eps) for _ in x]
             if optimizer == 'adam' else None)

    for _ in trange(n_steps, desc='Combined distance/material GD'):
        indices = np.flatnonzero(active)
        if not len(indices):
            break
        count = len(indices)
        distances, materials, distance_grads, material_grads = differentiable_optimization_tools(
            list(x[indices]), [edges] * count, [fixed_joints] * count,
            [motor] * count, target_curve, None,
        )
        gradient_values = np.column_stack((np.asarray(distances).reshape(-1),
                                           np.asarray(materials).reshape(-1)))

        # Gradient of the same combined loss, with fixed normalization scales.
        combined_gradient = np.zeros_like(x[indices])
        if coefficients[0] > 0:
            combined_gradient += coefficients[0] * np.asarray(distance_grads, dtype=float)
        if coefficients[1] > 0:
            combined_gradient += coefficients[1] * np.asarray(material_grads, dtype=float)
        usable = (np.isfinite(gradient_values).all(axis=1)
                  & np.isfinite(combined_gradient).all(axis=(1, 2)))
        active[indices[~usable]] = False
        indices = indices[usable]
        combined_gradient = combined_gradient[usable]
        if not len(indices):
            continue

        direction = -learning_rate * combined_gradient
        proposed_adams = {}
        if adams is not None:
            for j, i in enumerate(indices):
                trial_adam = copy.deepcopy(adams[i])
                proposed = trial_adam.step(x[i].copy(), combined_gradient[j]) - x[i]
                if np.isfinite(proposed).all() and np.sum(combined_gradient[j] * proposed) < 0:
                    direction[j] = proposed
                    proposed_adams[i] = trial_adam

        current_loss = combined_gd_loss(objectives[indices], scales, distance_weight)
        pending = np.arange(len(indices))
        step_scale = 1.0
        for _ in range(max_backtracks + 1):
            if not len(pending):
                break
            ids = indices[pending]
            trial_x = np.clip(x[ids] + step_scale * direction[pending], *bounds)
            trial_objectives = evaluate(trial_x)
            trial_loss = combined_gd_loss(trial_objectives, scales, distance_weight)
            slope = np.sum(combined_gradient[pending] * (trial_x - x[ids]), axis=(1, 2))
            accepted = (
                valid(trial_objectives) & np.isfinite(trial_x).all(axis=(1, 2))
                & (slope < 0) & (trial_loss < current_loss[pending])
                & (trial_loss <= current_loss[pending] + 1e-4 * slope)
            )
            accepted_ids = ids[accepted]
            x[accepted_ids] = trial_x[accepted]
            objectives[accepted_ids] = trial_objectives[accepted]
            if adams is not None:
                for i in accepted_ids:
                    if i in proposed_adams:
                        adams[i] = proposed_adams[i]
            pending = pending[~accepted]
            step_scale *= 0.5
        # An unusable gradient or rejected trial keeps the last valid coordinates.
        active[indices[pending]] = False
    return x

## First GA


In [ ]:
problem = mechanism_Synthesis_optimization(
    edges, fixed_joints, motor, target_curve, reference_point,
    lambda_max=LAMBDA_MAX,
)

X = np.repeat(x0[None], population_size, axis=0).reshape(population_size, -1)
algorithm = NSGA2(
    pop_size=population_size,
    sampling=X,
    crossover=SBX(prob=1.0, eta=3.0),
    mutation=PolynomialMutation(eta=20, prob=0.9),
    eliminate_duplicates=True,
)
results = minimize(
    problem, algorithm, ('n_gen', num_generations),
    verbose=True, save_history=True, seed=SEED,
)
if results.X is None:
    raise RuntimeError('The first GA found no feasible designs. GD needs a feasible GA population.')
results.scoring_F = problem.scoring_objectives(results.X)
initial_x = np.asarray(results.X).reshape(-1, len(x0), 2).copy()
print(f'First GA returned {len(initial_x)} designs.')

## Tune GD


In [ ]:
def gradient_descent_objective(trial):
    x_trial = refine_gradient_population(
        initial_x, edges, fixed_joints, motor, target_curve, reference_point,
        optimizer='gradient_descent',
        distance_weight=GD_DISTANCE_WEIGHT, max_backtracks=gd_max_backtracks,
        learning_rate=trial.suggest_float(
            'learning_rate', *learning_rate_range, log=True,
        ),
        n_steps=tuning_steps,
    )
    distances, materials = optimization_tools(
        list(x_trial), [edges] * len(x_trial), [fixed_joints] * len(x_trial),
        [motor] * len(x_trial), target_curve, target_idx=None,
    )
    objectives = np.column_stack((distances, materials))
    valid = np.isfinite(objectives).all(axis=1) & (objectives <= reference_point).all(axis=1)
    return float(HV(ref_point=reference_point)(objectives[valid])) if valid.any() else 0.0

study = optuna.create_study(
    direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(gradient_descent_objective, n_trials=tuning_trials)
print('Best tuning hypervolume (diagnostic):', study.best_value)
print('Best GD parameters:', study.best_params)

## Run GD


In [ ]:
x = refine_gradient_population(
    initial_x, edges, fixed_joints, motor, target_curve, reference_point,
    optimizer='gradient_descent', n_steps=tuning_steps,
    distance_weight=GD_DISTANCE_WEIGHT, max_backtracks=gd_max_backtracks,
    **study.best_params,
)
gd_x = x.copy()
print(f'GD returned {len(gd_x)} designs for the second GA.')

## Second GA


In [ ]:
gd_ga_algorithm = NSGA2(
    pop_size=len(gd_x), sampling=gd_x.reshape(len(gd_x), -1),
    crossover=SBX(prob=1.0, eta=3.0),
    mutation=PolynomialMutation(eta=1.0, prob=1.0),
    eliminate_duplicates=True,
)
gd_ga_results = minimize(
    problem, gd_ga_algorithm, ('n_gen', gd_ga_generations),
    verbose=True, save_history=True, seed=SEED,
)
gd_ga_results.scoring_F = (
    problem.scoring_objectives(gd_ga_results.X)
    if gd_ga_results.X is not None else None
)
if gd_ga_results.X is None:
    print('The second GA did not return feasible designs.')
else:
    print(f'Second GA returned {len(gd_ga_results.X)} designs.')

## Stage Results


In [ ]:
gd_distances, gd_materials = optimization_tools(
    list(gd_x), [edges] * len(gd_x), [fixed_joints] * len(gd_x),
    [motor] * len(gd_x), target_curve, target_idx=None,
)
gd_objectives = np.column_stack((gd_distances, gd_materials))
initial_gd_loss = combined_gd_loss(results.scoring_F, reference_point, GD_DISTANCE_WEIGHT)
final_gd_loss = combined_gd_loss(gd_objectives, reference_point, GD_DISTANCE_WEIGHT)
print(f'Mean combined GD loss: {initial_gd_loss.mean():.6f} → {final_gd_loss.mean():.6f}')
gd_valid = np.isfinite(gd_objectives).all(axis=1) & (gd_objectives <= reference_point).all(axis=1)
final_objectives = (
    gd_ga_results.scoring_F if gd_ga_results.scoring_F is not None
    else np.empty((0, 2))
)
stage_objectives = {
    'First GA': results.scoring_F,
    'GD': gd_objectives[gd_valid],
    'Second GA': final_objectives,
}
hv = HV(ref_point=reference_point)
for name, objectives in stage_objectives.items():
    stage_hv = float(hv(objectives)) if len(objectives) else 0.0
    print(f'{name}: {len(objectives)} designs; diagnostic HV = {stage_hv:.6f}')

if SHOW_PLOTS:
    fig, ax = plt.subplots(figsize=(6, 5))
    for name, objectives in stage_objectives.items():
        ax.scatter(objectives[:, 0], objectives[:, 1], label=name)
    ax.axvline(reference_point[0], color='gray', linestyle='--')
    ax.axhline(reference_point[1], color='gray', linestyle='--')
    ax.set(xlabel='Distance', ylabel='Material', title='GA → GD → GA')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Final Curve


In [ ]:
if gd_ga_results.X is not None:
    best_idx = int(np.argmin(gd_ga_results.scoring_F[:, 0]))
    best_sol = gd_ga_results.X[best_idx].reshape(-1, 2)
    print(f'Best final distance: {gd_ga_results.scoring_F[best_idx, 0]:.6f}')
    print(f'Material for that design: {gd_ga_results.scoring_F[best_idx, 1]:.6f}')
    if SHOW_PLOTS:
        solutions = np.asarray(PROBLEM_TOOLS.solver(
            [best_sol], [edges], [fixed_joints], [motor],
        ))
        traced_curve = solutions[0, problem.target_idx]
        COST_ENGINE.visualize_comparison(traced_curve, target_curve)

## Save and Score


In [ ]:
submission = make_empty_submission()
if gd_ga_results.X is not None:
    for member in gd_ga_results.X:
        submission[f'Problem {target_index + 1}'].append({
            'x0': member.reshape(-1, 2).copy(),
            'edges': edges.copy(),
            'fixed_joints': fixed_joints.copy(),
            'motor': motor.copy(),
            'target_joint': problem.target_idx,
        })

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = OUTPUT_DIR / f'problem_{target_index + 1}_ga_gd_ga.npy'
np.save(submission_path, submission)
scores = evaluate_submission(
    submission=str(submission_path), target_curves=str(TARGET_PATH),
)
pprint(scores)
print('Saved:', submission_path)
print('This file contains the selected target; the other two targets are empty.')